# 🔬 Visual Anomaly Detection — Part 1: Explore & Train

**Goal**: Understand anomaly detection fundamentals and train PatchCore, STFPM, and PaDiM on MVTec AD.

**Runtime**: Make sure you're using a **GPU runtime** → `Runtime > Change runtime type > T4 GPU`

---

### Key Concepts
- **One-class learning**: Train on ONLY normal images. Anything different = anomaly.
- **PatchCore**: Stores patch features in a memory bank → nearest-neighbor anomaly scoring
- **STFPM**: Student-teacher feature pyramid matching
- **MVTec AD**: 15 industrial categories (bottles, screws, carpets, etc.) with pixel-level anomaly masks

## 1️⃣ Setup & Installation

In [ ]:
# Install Anomalib + dependencies
!pip install -q anomalib torch torchvision opencv-python matplotlib seaborn scikit-learn tqdm pandas tabulate

In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

import anomalib
print(f"Anomalib version: {anomalib.__version__}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11

import cv2
from pathlib import Path
from tqdm.auto import tqdm

## 2️⃣ Load & Explore MVTec AD Dataset

MVTec AD has **15 categories** (10 objects + 5 textures). Each category has:
- **Training set**: Only normal (defect-free) images
- **Test set**: Normal + anomalous images with pixel-level ground truth masks

The dataset auto-downloads on first use (~350MB per category).

In [ ]:
from anomalib.data import MVTecAD
from torchvision.transforms.v2 import Resize

# All MVTec AD categories
MVTEC_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper",
]

OBJECT_CATEGORIES = ["bottle", "cable", "capsule", "hazelnut", "metal_nut",
                     "pill", "screw", "toothbrush", "transistor", "zipper"]
TEXTURE_CATEGORIES = ["carpet", "grid", "leather", "tile", "wood"]

print(f"Total categories: {len(MVTEC_CATEGORIES)}")
print(f"  Objects:  {len(OBJECT_CATEGORIES)} — {OBJECT_CATEGORIES}")
print(f"  Textures: {len(TEXTURE_CATEGORIES)} — {TEXTURE_CATEGORIES}")

In [ ]:
# Load the 'bottle' category (downloads ~350MB on first run)
CATEGORY = "bottle"  # ← Change this to try other categories!

datamodule = MVTecAD(
    root="./datasets/MVTecAD",
    category=CATEGORY,
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2,
    augmentations=Resize((256, 256)),
)
datamodule.setup()

print(f"\n✅ Dataset loaded: MVTec AD / {CATEGORY}")
print(f"   Training samples (normal only): {len(datamodule.train_data)}")
print(f"   Test samples (normal + anomalous): {len(datamodule.test_data)}")

In [ ]:
# Count normal vs anomalous in test set
test_labels = []
for i in range(len(datamodule.test_data)):
    sample = datamodule.test_data[i]
    label = sample["label"]
    if hasattr(label, "item"):
        label = label.item()
    test_labels.append(label)

n_normal = sum(1 for l in test_labels if l == 0)
n_anomalous = sum(1 for l in test_labels if l == 1)
print(f"Test set breakdown:")
print(f"  Normal:    {n_normal}")
print(f"  Anomalous: {n_anomalous}")
print(f"  Anomaly ratio: {n_anomalous/(n_normal+n_anomalous)*100:.1f}%")

### 🖼️ Visualize Normal vs Anomalous Samples

In [ ]:
# Separate normal and anomalous test indices
normal_indices = [i for i, l in enumerate(test_labels) if l == 0]
anomalous_indices = [i for i, l in enumerate(test_labels) if l == 1]

rng = np.random.default_rng(42)
n_show = 4

normal_pick = rng.choice(normal_indices, size=min(n_show, len(normal_indices)), replace=False)
anomalous_pick = rng.choice(anomalous_indices, size=min(n_show, len(anomalous_indices)), replace=False)

fig, axes = plt.subplots(3, n_show, figsize=(16, 12))
fig.suptitle(f"MVTec AD — {CATEGORY.title()}: Normal vs Anomalous", fontsize=16, fontweight="bold")

# Row 0: Normal
for i, idx in enumerate(normal_pick):
    sample = datamodule.test_data[idx]
    img = sample["image"].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    axes[0, i].imshow(img)
    axes[0, i].set_title("✅ Normal", color="green", fontweight="bold")
    axes[0, i].axis("off")

# Row 1: Anomalous images
# Row 2: Ground truth defect masks
for i, idx in enumerate(anomalous_pick):
    sample = datamodule.test_data[idx]
    img = sample["image"].permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)

    axes[1, i].imshow(img)
    axes[1, i].set_title("❌ Anomalous", color="red", fontweight="bold")
    axes[1, i].axis("off")

    if "mask" in sample:
        mask = sample["mask"].squeeze().numpy()
        # Red overlay on defect regions
        overlay = img.copy()
        mask_rgb = np.zeros_like(overlay)
        mask_rgb[:, :, 0] = mask
        overlay = cv2.addWeighted(overlay.astype(np.float32), 0.6,
                                  mask_rgb.astype(np.float32), 0.4, 0)
        axes[2, i].imshow(np.clip(overlay, 0, 1))
        axes[2, i].set_title("Defect Overlay", fontsize=10)
        axes[2, i].axis("off")

axes[0, 0].set_ylabel("Normal", fontsize=13, fontweight="bold")
axes[1, 0].set_ylabel("Anomalous", fontsize=13, fontweight="bold")
axes[2, 0].set_ylabel("GT Mask", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3️⃣ Train PatchCore (Memory Bank Approach)

**How PatchCore works:**
1. Extract patch features from a pretrained CNN (WideResNet-50)
2. Store representative features in a **memory bank** (coreset sampling)
3. At inference: compare each patch to nearest neighbor in memory → high distance = anomaly

**Key advantage**: Single-pass training (no gradient descent!) — very fast.

In [ ]:
from anomalib.models import Patchcore
from anomalib.engine import Engine

# Create PatchCore model
patchcore_model = Patchcore(
    backbone="wide_resnet50_2",
    layers=("layer2", "layer3"),    # Multi-scale feature extraction
    coreset_sampling_ratio=0.1,      # Keep 10% of patches in memory bank
    num_neighbors=9,                 # k-NN for scoring
)

# Create datamodule
patchcore_data = MVTecAD(
    root="./datasets/MVTecAD",
    category=CATEGORY,
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2,
    augmentations=Resize((256, 256)),
)

print("🚀 Training PatchCore...")
print("   (This is single-pass — should be fast even on T4!)")

In [ ]:
# Train!
patchcore_engine = Engine(
    max_epochs=1,  # PatchCore needs only 1 epoch
    accelerator="auto",
    devices=1,
    default_root_dir="./results/patchcore",
)

patchcore_engine.fit(model=patchcore_model, datamodule=patchcore_data)
print("\n✅ PatchCore training complete!")

In [ ]:
# Test and get metrics
patchcore_results = patchcore_engine.test(
    model=patchcore_model,
    datamodule=patchcore_data,
)

print("\n📊 PatchCore Results:")
for metric_dict in patchcore_results:
    for k, v in metric_dict.items():
        if isinstance(v, float):
            print(f"   {k}: {v:.4f}")

### 🗺️ Visualize PatchCore Predictions

Let's see the anomaly heatmaps PatchCore generates!

In [ ]:
# Get predictions on test set
patchcore_predictions = patchcore_engine.predict(
    model=patchcore_model,
    datamodule=patchcore_data,
)

print(f"Got {len(patchcore_predictions)} prediction batches")

In [ ]:
def visualize_predictions(predictions, title="Anomaly Detection Results", n_show=8):
    """Visualize anomaly detection predictions with heatmaps."""
    # Collect all samples from batches
    all_images, all_maps, all_labels, all_scores = [], [], [], []

    for batch in predictions:
        images = batch.image
        anomaly_maps = batch.anomaly_map if hasattr(batch, 'anomaly_map') else None
        labels = batch.gt_label if hasattr(batch, 'gt_label') else None
        scores = batch.pred_score if hasattr(batch, 'pred_score') else None

        for i in range(len(images)):
            img = images[i].permute(1, 2, 0).cpu().numpy()
            all_images.append(np.clip(img, 0, 1))

            if anomaly_maps is not None:
                amap = anomaly_maps[i].squeeze().cpu().numpy()
                all_maps.append(amap)

            if labels is not None:
                all_labels.append(int(labels[i].cpu().item()))

            if scores is not None:
                all_scores.append(float(scores[i].cpu().item()))

    # Pick samples to show (mix of normal + anomalous)
    n_show = min(n_show, len(all_images))
    # Try to show both normal and anomalous
    if all_labels:
        normal_idx = [i for i, l in enumerate(all_labels) if l == 0]
        anom_idx = [i for i, l in enumerate(all_labels) if l == 1]
        rng = np.random.default_rng(42)
        pick_n = min(n_show // 2, len(normal_idx))
        pick_a = min(n_show - pick_n, len(anom_idx))
        indices = list(rng.choice(normal_idx, pick_n, replace=False)) + \
                  list(rng.choice(anom_idx, pick_a, replace=False))
    else:
        indices = list(range(n_show))

    fig, axes = plt.subplots(2, len(indices), figsize=(3.5 * len(indices), 7))
    if len(indices) == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(title, fontsize=15, fontweight="bold")

    for col, idx in enumerate(indices):
        img = all_images[idx]
        label = all_labels[idx] if all_labels else -1
        score = all_scores[idx] if all_scores else 0

        # Original image
        label_str = "❌ Anomalous" if label == 1 else "✅ Normal"
        color = "red" if label == 1 else "green"
        axes[0, col].imshow(img)
        axes[0, col].set_title(f"{label_str}\nScore: {score:.3f}", color=color, fontsize=9, fontweight="bold")
        axes[0, col].axis("off")

        # Heatmap overlay
        if all_maps:
            amap = all_maps[idx]
            if amap.max() > amap.min():
                amap_norm = (amap - amap.min()) / (amap.max() - amap.min())
            else:
                amap_norm = np.zeros_like(amap)
            heatmap = cv2.applyColorMap((amap_norm * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
            overlay = cv2.addWeighted((img * 255).astype(np.uint8), 0.6,
                                      (heatmap_rgb * 255).astype(np.uint8), 0.4, 0)
            axes[1, col].imshow(overlay)
            axes[1, col].set_title("Anomaly Heatmap", fontsize=9)
            axes[1, col].axis("off")

    axes[0, 0].set_ylabel("Input", fontsize=12, fontweight="bold")
    axes[1, 0].set_ylabel("Heatmap", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


visualize_predictions(patchcore_predictions, f"PatchCore — {CATEGORY.title()}", n_show=8)

## 4️⃣ Train STFPM (Student-Teacher Feature Pyramid)

**How STFPM works:**
1. **Teacher**: Pretrained CNN (frozen) — knows general features
2. **Student**: Trained to mimic teacher on normal images only
3. At inference: compare student vs teacher features → mismatch = anomaly

**Key advantage**: Fast inference, good for real-time applications.

In [ ]:
from anomalib.models import Stfpm

stfpm_model = Stfpm(
    backbone="resnet18",
    layers=("layer1", "layer2", "layer3"),
)

stfpm_data = MVTecAD(
    root="./datasets/MVTecAD",
    category=CATEGORY,
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2,
    augmentations=Resize((256, 256)),
)

stfpm_engine = Engine(
    max_epochs=100,
    accelerator="auto",
    devices=1,
    default_root_dir="./results/stfpm",
)

print("🚀 Training STFPM (100 epochs)...")
stfpm_engine.fit(model=stfpm_model, datamodule=stfpm_data)
print("\n✅ STFPM training complete!")

In [ ]:
# Test STFPM
stfpm_results = stfpm_engine.test(model=stfpm_model, datamodule=stfpm_data)
print("\n📊 STFPM Results:")
for metric_dict in stfpm_results:
    for k, v in metric_dict.items():
        if isinstance(v, float):
            print(f"   {k}: {v:.4f}")

# Visualize
stfpm_predictions = stfpm_engine.predict(model=stfpm_model, datamodule=stfpm_data)
visualize_predictions(stfpm_predictions, f"STFPM — {CATEGORY.title()}", n_show=8)

## 5️⃣ Compare Models

Let's see how PatchCore and STFPM stack up side by side.

In [ ]:
import pandas as pd

# Collect results
comparison = {}
for name, results in [("PatchCore", patchcore_results), ("STFPM", stfpm_results)]:
    metrics = {}
    for d in results:
        for k, v in d.items():
            if isinstance(v, float):
                metrics[k] = round(v, 4)
    comparison[name] = metrics

df = pd.DataFrame(comparison).T
print(f"\n🏆 Model Comparison — MVTec AD / {CATEGORY.title()}")
print("="*60)
print(df.to_string())

# Bar chart
if "image_AUROC" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    models = df.index.tolist()
    auroc_vals = df["image_AUROC"].values
    colors = ["#2196F3", "#4CAF50"]
    bars = ax.bar(models, auroc_vals, color=colors[:len(models)], edgecolor="white", linewidth=2)
    for bar, val in zip(bars, auroc_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
    ax.set_ylabel("Image AUROC")
    ax.set_title(f"Model Comparison — {CATEGORY.title()}", fontweight="bold", fontsize=14)
    ax.set_ylim(0, 1.1)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6️⃣ Try Different Categories!

Go back to the top and change `CATEGORY` to try:
- `"carpet"` — texture with various defect types
- `"hazelnut"` — small surface cracks/holes
- `"screw"` — thread damage, scratches
- `"tile"` — cracks and glue strips

---

### 🎓 Interview Questions This Covers

| Question | You Can Now Answer |
|----------|-------------------|
| What is one-class classification? | Train on normal only, detect deviations |
| How does PatchCore work? | Memory bank of coreset patch features + k-NN |
| What is student-teacher distillation? | STFPM: student mimics teacher on normal, discrepancy = anomaly |
| What metrics are used for anomaly detection? | Image AUROC, Pixel AUROC, F1 |
| What is MVTec AD? | 15-category industrial anomaly benchmark |

**Next notebook**: Build a custom autoencoder from scratch! → `02_autoencoder_from_scratch.ipynb`